# 布朗运动与随机过程可视化

本 Notebook 演示标准布朗运动、几何布朗运动（GBM）与蒙特卡洛路径模拟。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 1. 标准布朗运动（Wiener Process）

$W_t = W_0 + \sum_{i=1}^{t} \epsilon_i$，其中 $\epsilon_i \sim N(0, \Delta t)$

In [ ]:
T, dt, n_paths = 1.0, 1/252, 10
steps = int(T / dt)
t = np.linspace(0, T, steps)

paths = np.zeros((steps, n_paths))
for i in range(1, steps):
    paths[i] = paths[i-1] + np.random.normal(0, np.sqrt(dt), n_paths)

plt.figure()
plt.plot(t, paths, alpha=0.6, linewidth=0.8)
plt.axhline(0, color='black', linewidth=1)
plt.fill_between(t, -2*np.sqrt(t), 2*np.sqrt(t), alpha=0.1, color='blue', label='±2σ 置信区间')
plt.title('标准布朗运动 (10条路径)')
plt.xlabel('时间 (年)')
plt.ylabel('W(t)')
plt.legend()
plt.show()

## 2. 几何布朗运动（股票价格模型）

$dS_t = \mu S_t dt + \sigma S_t dW_t$

解析解：$S_t = S_0 \exp\left[(\mu - \frac{\sigma^2}{2})t + \sigma W_t\right]$

In [ ]:
S0, mu, sigma = 100, 0.10, 0.20
n_paths = 50

gbm = np.zeros((steps, n_paths))
gbm[0] = S0
for i in range(1, steps):
    z = np.random.normal(0, 1, n_paths)
    gbm[i] = gbm[i-1] * np.exp((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*z)

# 理论均值与置信区间
mean_path = S0 * np.exp(mu * t)
upper = S0 * np.exp((mu + 2*sigma) * t)
lower = S0 * np.exp((mu - 2*sigma) * t)

plt.figure()
plt.plot(t, gbm, alpha=0.15, linewidth=0.5, color='steelblue')
plt.plot(t, mean_path, 'r-', linewidth=2, label=f'理论均值 (μ={mu})')
plt.fill_between(t, lower, upper, alpha=0.15, color='red', label='±2σ 区间')
plt.title(f'几何布朗运动 (S₀={S0}, μ={mu}, σ={sigma}, {n_paths}条路径)')
plt.xlabel('时间 (年)')
plt.ylabel('价格')
plt.legend()
plt.show()

print(f'终值均值: {gbm[-1].mean():.2f} (理论: {mean_path[-1]:.2f})')
print(f'终值中位数: {np.median(gbm[-1]):.2f}')

## 3. 收益率分布检验

验证 GBM 下日对数收益率服从正态分布。

In [ ]:
from scipy import stats

log_returns = np.diff(np.log(gbm[:, 0]))  # 取第一条路径

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 直方图 vs 正态分布
axes[0].hist(log_returns, bins=30, density=True, alpha=0.7, color='steelblue', label='实际分布')
x = np.linspace(log_returns.min(), log_returns.max(), 100)
axes[0].plot(x, stats.norm.pdf(x, log_returns.mean(), log_returns.std()), 'r-', lw=2, label='正态分布')
axes[0].set_title('日对数收益率分布')
axes[0].legend()

# Q-Q 图
stats.probplot(log_returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q 图')

plt.tight_layout()
plt.show()

jb_stat, jb_p = stats.jarque_bera(log_returns)
print(f'JB 检验: stat={jb_stat:.4f}, p={jb_p:.4f} → {"正态" if jb_p > 0.05 else "非正态"}')